# Rheology Data Schema Design

This notebook develops a canonical schema for rheological measurements
of hydrogels made from bovine (cattle-derived) collagen. It builds on
the completed workbook audit to organise processed data consistently
while preserving experimental context and source provenance.

<details>
<summary><strong>Notebook scope and schema-design principles</strong></summary>

## Scope and audit basis

The completed audit covered 20 source workbooks containing
660 measurement rows and recorded eight findings in
`data/quality_control/data_quality_issue_log.csv`.

The audit's working expected schemas describe the source workbook
layouts. This notebook develops the structure for the processed
data, including table definitions, identifiers, field names,
data types, units and validation requirements.

## Proposed data structure

Each table represents a distinct level of information.
Table granularity defines what one record represents.

| Table | Record definition |
|---|---|
| `samples` | One physical hydrogel replicate, described by its concentration and original replicate label. Any provisional identity assignment must be explicitly documented. |
| `experiments` | One rheological test represented by a source workbook and worksheet, with its test type, available experimental metadata and sample link where supported. |
| `measurements` | One measurement point within an experiment, retaining the source-reported variables and original Excel row number. |
| `quality_issues` | One documented finding, identified by an issue identifier and linked to the affected source workbook, worksheet and cell or range. |

This separation supports traceable relationships without repeating
sample-level information in every measurement record.

## Experimental unit and sample identity

The physical hydrogel replicate is the experimental unit.
Frequency points and sequential observations within a test are
repeated measurements, not independent hydrogel samples.

Sample identity must be distinguished from filename correspondence.
Matching concentration and replicate labels across test types will
not, by themselves, be treated as confirmation that the same
physical hydrogel was tested.

The sample-linkage review below documents these associations and
their implications for identifier design.

## Measurement coordinates and experimental metadata

The source time-sweep workbooks contain `Meas. Pts.` but no explicit
elapsed-time variable or time unit. This field will therefore retain
its meaning as measurement-point order. Elapsed time, durations and
time-dependent rates will not be inferred without supporting
timing metadata.

Reported experimental settings will remain distinguishable from
measurement-level values recorded in the workbooks.

## Provenance and transformation rules

- Preserve original Excel workbooks without modification.
- Retain original filenames, worksheet names and source-row locations.
- Preserve source variable headings and unit labels alongside
  documented standardised mappings.
- Distinguish source-reported quantities from quantities calculated
  during processing.
- Identify known instrument-derived quantities explicitly; a numeric
  source cell does not establish that a quantity was directly measured.
- Retain uncertain observations with appropriate quality-control flags.
- Link exclusions and corrections to documented processing decisions.

## Current design status

The draft experiment register and the experiment and measurement data
dictionaries were constructed and exported. Structural checks were
implemented for the experiment register.

Physical sample linkage remains unresolved. Measurement extraction,
measurement-level quality-control linkage and enforcement of the
proposed measurement rules are reserved for subsequent processing.

</details>

## Sample-Linkage Review

This review identifies source workbooks with matching concentration
and replicate labels to inform sample and experiment identifier design.

<details>
<summary><strong>Method, findings and sample-linkage decision</strong></summary>

### Method

The existing file inventory,
`metadata/extracted_file_inventory.csv`, is grouped by the combined
values of:

- `dataset_id`;
- `collagen_concentration_mg_ml`; and
- `replicate_id`.

The dataset identifier keeps the grouping specific to its source
dataset. For each group, the code counts the workbooks and lists
their test types and original filenames.

The `cross_test_identity_status` field records the current
interpretation: matching filename labels alone do not confirm
shared physical sample identity. This status is an explicit
curation decision, not an experimental result inferred by the code.

### Findings

The review identifies 10 concentration–replicate label groups
covering all 20 source workbooks.

| Collagen concentration (mg/mL) | Label groups | Frequency-sweep workbooks | Time-sweep workbooks |
|---|---:|---:|---:|
| 0.8 | 4 | 4 | 4 |
| 1.5 | 3 | 3 | 3 |
| 2.3 | 3 | 3 | 3 |
| **Total** | **10** | **10** | **10** |

Each group contains one frequency-sweep workbook and one time-sweep
workbook with matching concentration and replicate labels.

### Evidence boundary

The grouping establishes correspondence between filenames.
It does not independently verify the number of unique physical
hydrogels or whether the two test types used the same specimens.

For example, the 0.8 mg/mL workbooks contain replicate labels 1–4
for both test types. These eight workbooks cannot automatically
be interpreted as either four or eight unique physical samples.

### Schema-design decision

Retain the filename-based associations for traceability while
keeping the two tests as separate experiment records.

Any shared physical sample identifier across test types will
require documented evidence or an explicitly declared assumption.
Until this relationship is resolved, matching labels will not
justify paired cross-test analysis.

### Execution and data preservation

The code reads the existing inventory and creates an in-memory
review table. It does not modify source files, merge measurement
records or save additional files.

</details>

In [3]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# Locate the project folder.
current_folder = Path.cwd().resolve()
project_root = (
    current_folder.parent
    if current_folder.name.lower() in {"notebook", "notebooks"}
    else current_folder
)

# Read the existing audit inventory without changing it.
inventory = pd.read_csv(
    project_root / "metadata" / "extracted_file_inventory.csv",
    encoding="utf-8-sig",
)

# Group source files by their recorded concentration and replicate label.
sample_linkage_review = (
    inventory.groupby(
        ["dataset_id", "collagen_concentration_mg_ml", "replicate_id"],
        as_index=False,
        dropna=False,
    )
    .agg(
        workbook_count=("original_filename", "size"),
        test_types=(
            "test_type_standard",
            lambda values: " | ".join(sorted(values)),
        ),
        source_workbooks=(
            "original_filename",
            lambda values: " | ".join(sorted(values)),
        ),
    )
)

# Record uncertainty rather than assume shared physical identity.
sample_linkage_review["cross_test_identity_status"] = (
    "Unconfirmed: matching filename labels only"
)

# Print the summary.
print("SAMPLE-LINKAGE REVIEW")
print(f"Concentration–replicate groups: {len(sample_linkage_review)}")
print(
    "Source workbooks represented: "
    f"{sample_linkage_review['workbook_count'].sum()}"
)
print()

# Display a formatted table with all rows, columns and complete filenames.
with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
):
    display(sample_linkage_review)

SAMPLE-LINKAGE REVIEW
Concentration–replicate groups: 10
Source workbooks represented: 20



,dataset_id,collagen_concentration_mg_ml,replicate_id,workbook_count,test_types,source_workbooks,cross_test_identity_status
0,zenodo_17413651,0.8,1,2,frequency_sweep | time_sweep,Colageno_bov_0.8_1_frecuencia.xlsx | Colageno_bov_0.8_1_tiempo.xlsx,Unconfirmed: matching filename labels only
1,zenodo_17413651,0.8,2,2,frequency_sweep | time_sweep,Colageno_bov_0.8_2_frecuencia.xlsx | Colageno_bov_0.8_2_tiempo.xlsx,Unconfirmed: matching filename labels only
2,zenodo_17413651,0.8,3,2,frequency_sweep | time_sweep,Colageno_bov_0.8_3_frecuencia.xlsx | Colageno_bov_0.8_3_tiempo.xlsx,Unconfirmed: matching filename labels only
3,zenodo_17413651,0.8,4,2,frequency_sweep | time_sweep,Colageno_bov_0.8_4_frecuencia.xlsx | Colageno_bov_0.8_4_tiempo.xlsx,Unconfirmed: matching filename labels only
4,zenodo_17413651,1.5,1,2,frequency_sweep | time_sweep,Colageno_bov_1.5_1_frecuencia.xlsx | Colageno_bov_1.5_1_tiempo.xlsx,Unconfirmed: matching filename labels only
5,zenodo_17413651,1.5,2,2,frequency_sweep | time_sweep,Colageno_bov_1.5_2_frecuencia.xlsx | Colageno_bov_1.5_2_tiempo.xlsx,Unconfirmed: matching filename labels only
6,zenodo_17413651,1.5,3,2,frequency_sweep | time_sweep,Colageno_bov_1.5_3_frecuencia.xlsx | Colageno_bov_1.5_3_tiempo.xlsx,Unconfirmed: matching filename labels only
7,zenodo_17413651,2.3,1,2,frequency_sweep | time_sweep,Colageno_bov_2.3_1_frecuencia.xlsx | Colageno_bov_2.3_1_tiempo.xlsx,Unconfirmed: matching filename labels only
8,zenodo_17413651,2.3,2,2,frequency_sweep | time_sweep,Colageno_bov_2.3_2_frecuencia.xlsx | Colageno_bov_2.3_2_tiempo.xlsx,Unconfirmed: matching filename labels only
9,zenodo_17413651,2.3,3,2,frequency_sweep | time_sweep,Colageno_bov_2.3_3_frecuencia.xlsx | Colageno_bov_2.3_3_tiempo.xlsx,Unconfirmed: matching filename labels only


## Draft Experiment Register

A draft experiment register was constructed to identify each recorded rheological test and link it to its source workbook and worksheet. This provides experiment-level traceability without assuming that matching replicate labels across test types represent the same physical hydrogel.

<details>
<summary><strong>Source linkage, experiment identifiers and validation checks</strong></summary>

### Register construction

The code combined the experiment metadata in
`metadata/extracted_file_inventory.csv` with the worksheet names in
`metadata/workbook_structure_inventory.csv`, using `original_filename`
as the matching key.

The register retained each workbook’s dataset identifier, collagen
concentration, source replicate label and standardised test type.
The merge required a one-to-one relationship between the filename
records in the two inventories. This design assumes one audited
worksheet per workbook.

### Experiment identifiers

Each `experiment_id` combined three source identifiers:

`dataset_id::original_filename::sheet_name`

These identifiers remain unchanged when the table is reordered or the
code is rerun, provided that the dataset identifier, filename and
worksheet name remain unchanged. Renaming any of these components
changes the identifier. The identifier does not detect changes to
workbook contents.

### Observed results and implemented checks

The executed code produced **20 experiment records with 20 unique
experiment identifiers**.

The code checked that:

- Filename matching satisfied the one-to-one merge requirement.
- Every registered workbook had a non-missing worksheet name.
- No duplicate experiment identifiers were generated.

These checks established structural consistency for the register.
They did not validate the rheological measurements or confirm physical
sample identity.

### Interpretation of experiment records

Each record represents a source-recorded test, rather than a confirmed
independent hydrogel sample. Frequency-sweep and time-sweep records
remain separate experiments.

Matching concentration and replicate labels across these test types
were retained as contextual metadata. Whether the corresponding tests
used the same physical hydrogel remains unconfirmed.

### Execution dependencies and output

This cell requires the preceding setup and sample-linkage code to have
defined `pd`, `project_root`, `inventory` and `display`. After restarting
the kernel, those earlier cells must be executed first.

The code creates and displays `experiment_register` in memory. It does
not save the register to disk or modify the source workbooks or audit
metadata files.

</details>

In [5]:
# Construct a draft experiment register from the existing audit metadata.

workbook_structure = pd.read_csv(
    project_root / "metadata" / "workbook_structure_inventory.csv",
    encoding="utf-8-sig",
)

# Link each inventoried workbook to its audited worksheet.
experiment_register = inventory[
    [
        "dataset_id",
        "original_filename",
        "collagen_concentration_mg_ml",
        "replicate_id",
        "test_type_standard",
    ]
].merge(
    workbook_structure[["original_filename", "sheet_name"]],
    on="original_filename",
    how="left",
    validate="one_to_one",
)

# Stop if any workbook lacks an audited worksheet name.
if experiment_register["sheet_name"].isna().any():
    raise ValueError("An inventoried workbook has no matching worksheet record.")

# Build identifiers from source identity, rather than table row numbers.
experiment_register["experiment_id"] = (
    experiment_register["dataset_id"]
    + "::"
    + experiment_register["original_filename"]
    + "::"
    + experiment_register["sheet_name"]
)

# Confirm that every experiment identifier is unique.
if not experiment_register["experiment_id"].is_unique:
    raise ValueError("Duplicate experiment identifiers were generated.")

print("DRAFT EXPERIMENT REGISTER")
print(f"Experiment records: {len(experiment_register)}")
print(
    "Unique experiment identifiers: "
    f"{experiment_register['experiment_id'].nunique()}"
)

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
):
    display(
        experiment_register[
            ["experiment_id", "test_type_standard"]
        ]
    )

DRAFT EXPERIMENT REGISTER
Experiment records: 20
Unique experiment identifiers: 20


,experiment_id,test_type_standard
0,zenodo_17413651::Colageno_bov_0.8_1_frecuencia.xlsx::Hoja1,frequency_sweep
1,zenodo_17413651::Colageno_bov_0.8_1_tiempo.xlsx::Hoja1,time_sweep
2,zenodo_17413651::Colageno_bov_0.8_2_frecuencia.xlsx::Hoja1,frequency_sweep
3,zenodo_17413651::Colageno_bov_0.8_2_tiempo.xlsx::Hoja1,time_sweep
4,zenodo_17413651::Colageno_bov_0.8_3_frecuencia.xlsx::Hoja1,frequency_sweep
5,zenodo_17413651::Colageno_bov_0.8_3_tiempo.xlsx::Hoja1,time_sweep
6,zenodo_17413651::Colageno_bov_0.8_4_frecuencia.xlsx::Hoja1,frequency_sweep
7,zenodo_17413651::Colageno_bov_0.8_4_tiempo.xlsx::Hoja1,time_sweep
8,zenodo_17413651::Colageno_bov_1.5_1_frecuencia.xlsx::Hoja1,frequency_sweep
9,zenodo_17413651::Colageno_bov_1.5_1_tiempo.xlsx::Hoja1,time_sweep


## Experiment Register Data Dictionary

The draft data dictionary specifies the meaning, intended data type
and requirement status of each field in the experiment register.
It provides explicit field definitions for subsequent schema
implementation and validation.

<details>
<summary><strong>Field definitions and implementation status</strong></summary>

### Design choices

The register retains the dataset identifier, original workbook name
and worksheet name alongside the combined `experiment_id`. Keeping
these components separately supports direct source lookup without
requiring the combined identifier to be split.

Collagen concentration and replicate labels describe the source
records. Matching labels across test types do not establish shared
physical sample identity.

The proposed representation of `replicate_id` is text because it
functions as a label rather than a numerical measurement.
Collagen concentration is represented as a number in mg/mL.

### Scope of the specification

The dictionary describes the seven fields currently included in the
draft experiment register. It does not yet define the complete
experimental-metadata schema or assign physical sample identifiers.

The listed data types and requirement statuses are proposed
specifications. Creating this dictionary does not convert the
register's data types or enforce field-level validation.

### Execution and output

The following cell defines the dictionary directly in Python and
displays it as `experiment_data_dictionary`. It can run independently
of earlier in-memory objects, provided that pandas and IPython are
available.

The dictionary is created in memory only. No file is saved, and
the experiment register and source records remain unchanged.

</details>

In [7]:
import pandas as pd
from IPython.display import display

# Define the proposed fields of the current draft experiment register.
experiment_field_definitions = [
    (
        "experiment_id",
        "text",
        True,
        "Not applicable",
        "Identifier formed as dataset_id::original_filename::sheet_name.",
    ),
    (
        "dataset_id",
        "text",
        True,
        "Not applicable",
        "Project-assigned dataset label recorded during the source audit.",
    ),
    (
        "original_filename",
        "text",
        True,
        "Not applicable",
        "Original source workbook filename, including its extension.",
    ),
    (
        "sheet_name",
        "text",
        True,
        "Not applicable",
        "Audited worksheet name within the source workbook.",
    ),
    (
        "collagen_concentration_mg_ml",
        "number",
        True,
        "mg/mL",
        "Collagen concentration parsed from the source filename during the audit.",
    ),
    (
        "replicate_id",
        "text",
        True,
        "Not applicable",
        "Source replicate label; shared physical identity across tests is unconfirmed.",
    ),
    (
        "test_type_standard",
        "text",
        True,
        "Not applicable",
        "Standardised test label: frequency_sweep or time_sweep.",
    ),
]

experiment_data_dictionary = pd.DataFrame(
    experiment_field_definitions,
    columns=[
        "field_name",
        "proposed_data_type",
        "required",
        "unit",
        "definition",
    ],
)

print("DRAFT EXPERIMENT DATA DICTIONARY")
print(f"Fields documented: {len(experiment_data_dictionary)}")

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
):
    display(experiment_data_dictionary)

DRAFT EXPERIMENT DATA DICTIONARY
Fields documented: 7


,field_name,proposed_data_type,required,unit,definition
0,experiment_id,text,True,Not applicable,Identifier formed as dataset_id::original_filename::sheet_name.
1,dataset_id,text,True,Not applicable,Project-assigned dataset label recorded during the source audit.
2,original_filename,text,True,Not applicable,"Original source workbook filename, including its extension."
3,sheet_name,text,True,Not applicable,Audited worksheet name within the source workbook.
4,collagen_concentration_mg_ml,number,True,mg/mL,Collagen concentration parsed from the source filename during the audit.
5,replicate_id,text,True,Not applicable,Source replicate label; shared physical identity across tests is unconfirmed.
6,test_type_standard,text,True,Not applicable,Standardised test label: frequency_sweep or time_sweep.


## Experiment Register Field Checks

The draft experiment register is compared with its data dictionary
to identify missing fields, missing or blank values, and differences
between actual values and proposed data types.

<details>
<summary><strong>Check scope and interpretation</strong></summary>

The checks assess each documented field without modifying the register.
Text fields require string values. Number fields require real numerical
values, excluding Boolean values. Missing values and blank strings are
reported separately from data-type mismatches.

A type mismatch identifies a difference from the proposed representation.
For example, a replicate label stored as an integer does not meet the
proposed text specification, even if the label itself is correct.

These checks do not establish physical sample identity, verify source
measurements or enforce all future schema rules. Allowed test-type
values, numerical ranges and links between tables are outside this
cell's scope.

The cell requires `experiment_register` and
`experiment_data_dictionary` from the preceding cells. It creates
an in-memory check report and does not save files or convert values.

</details>

In [9]:
from numbers import Real
import pandas as pd
from IPython.display import display

field_check_records = []

for field in experiment_data_dictionary.itertuples(index=False):
    field_present = field.field_name in experiment_register.columns

    result = {
        "field_name": field.field_name,
        "proposed_data_type": field.proposed_data_type,
        "field_present": field_present,
        "missing_values": None,
        "blank_values": None,
        "type_mismatches": None,
    }

    if field_present:
        values = experiment_register[field.field_name]

        missing = values.isna()
        blank = values.map(
            lambda value: isinstance(value, str) and not value.strip()
        )

        # Check types only for values that are neither missing nor blank.
        populated_values = values.loc[~missing & ~blank]

        if field.proposed_data_type == "text":
            type_matches = populated_values.map(
                lambda value: isinstance(value, str)
            )
        elif field.proposed_data_type == "number":
            type_matches = populated_values.map(
                lambda value: (
                    isinstance(value, Real)
                    and not isinstance(value, bool)
                )
            )
        else:
            raise ValueError(
                f"Unsupported proposed type: {field.proposed_data_type}"
            )

        result.update(
            missing_values=int(missing.sum()),
            blank_values=int(blank.sum()),
            type_mismatches=int((~type_matches).sum()),
        )

    field_check_records.append(result)

experiment_field_checks = pd.DataFrame(field_check_records)

print("EXPERIMENT REGISTER FIELD CHECKS")
print(f"Experiment records inspected: {len(experiment_register)}")
print(f"Documented fields checked: {len(experiment_field_checks)}")

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
):
    display(experiment_field_checks)

EXPERIMENT REGISTER FIELD CHECKS
Experiment records inspected: 20
Documented fields checked: 7


,field_name,proposed_data_type,field_present,missing_values,blank_values,type_mismatches
0,experiment_id,text,True,0,0,0
1,dataset_id,text,True,0,0,0
2,original_filename,text,True,0,0,0
3,sheet_name,text,True,0,0,0
4,collagen_concentration_mg_ml,number,True,0,0,0
5,replicate_id,text,True,0,0,20
6,test_type_standard,text,True,0,0,0


## Replicate Label Representation

The field checks found all seven documented fields present across
20 experiment records, with no missing or blank values. The
`replicate_id` field contained 20 values that did not meet the proposed
text specification. No type mismatches were reported for the other
six fields.

<details>
<summary><strong>Representation decision and verification</strong></summary>

Replicate identifiers are source labels rather than numerical
measurements. The following operation represents them as text in a
separate register copy, `experiment_register_standardised`.

A before-and-after summary records the values and their Python types.
The code checks that the converted labels are non-missing, non-blank
strings and that all other columns remain unchanged.

This operation does not establish shared physical sample identity
across test types or correct an experimental observation.

The cell requires `experiment_register` from the preceding register
construction. After a kernel restart, the earlier setup and register
cells must be run first. Both registers remain in memory; no source
file or audit metadata file is modified, and no new file is saved.

</details>

In [11]:
import pandas as pd
from IPython.display import display

# Preserve the original register and standardise a separate copy.
experiment_register_standardised = experiment_register.copy(deep=True)

original_labels = experiment_register["replicate_id"]

if original_labels.isna().any():
    raise ValueError("Missing replicate labels must be reviewed first.")

experiment_register_standardised["replicate_id"] = (
    original_labels.astype("string")
)

standardised_labels = experiment_register_standardised["replicate_id"]

# Check the resulting label representation.
if standardised_labels.isna().any():
    raise ValueError("Missing labels were found after conversion.")

if standardised_labels.str.strip().eq("").any():
    raise ValueError("Blank replicate labels were found.")

if not standardised_labels.map(lambda value: isinstance(value, str)).all():
    raise TypeError("Some replicate labels are not strings.")

# Confirm that the conversion did not change any other column.
pd.testing.assert_frame_equal(
    experiment_register.drop(columns="replicate_id"),
    experiment_register_standardised.drop(columns="replicate_id"),
)

# Display each distinct before-and-after representation.
replicate_type_review = pd.DataFrame(
    {
        "original_value": original_labels,
        "original_python_type": original_labels.map(
            lambda value: type(value).__name__
        ),
        "standardised_value": standardised_labels,
        "standardised_python_type": standardised_labels.map(
            lambda value: type(value).__name__
        ),
    }
).drop_duplicates(ignore_index=True)

print("REPLICATE LABEL REPRESENTATION")
print(f"Experiment records retained: {len(experiment_register_standardised)}")
print("All replicate labels are non-missing, non-blank strings: True")
print("All other columns unchanged: True")
print("Files written: 0")

display(replicate_type_review)

REPLICATE LABEL REPRESENTATION
Experiment records retained: 20
All replicate labels are non-missing, non-blank strings: True
All other columns unchanged: True
Files written: 0


,original_value,original_python_type,standardised_value,standardised_python_type
0,1,int,1,str
1,2,int,2,str
2,3,int,3,str
3,4,int,4,str


## Measurement Identity and Row-Level Provenance

The proposed measurement table represents one measurement point per
record. Each record will link to its experiment and retain its original
Excel row number, allowing observations to be traced to their source.

<details>
<summary><strong>Identifier design and measurement coordinates</strong></summary>

### Measurement identifiers

The proposed `measurement_id` combines the experiment identifier with
the original Excel row number:

`experiment_id::row_<source_excel_row>`

This distinguishes individual observations within an experiment.
The identifier remains stable when processed records are reordered,
provided that the experiment identifier and original source-row
location remain unchanged. Inserting or deleting rows in the source
workbook can change these identifiers.

### Source row and measurement point

`source_excel_row` records the observation's physical row location
in Excel. `measurement_point` preserves the source-reported
`Meas. Pts.` value.

These fields are kept separately because a worksheet row number
includes preceding headings and unit rows, whereas the reported
measurement-point sequence describes observations within the test.

For time-sweep records, measurement-point order will not be treated
as elapsed time. Multiple points within a test remain repeated
observations rather than independent physical hydrogel samples.

### Specification status and execution

The following dictionary defines four proposed identity and provenance
fields. Rheological variables and their units will be specified
separately.

This cell creates and displays `measurement_identity_dictionary`
in memory. It does not extract measurements, generate measurement
identifiers, enforce validation rules or save files. It requires
pandas and IPython but does not depend on earlier in-memory tables.

</details>

In [13]:
import pandas as pd
from IPython.display import display

# Specify the proposed identity and provenance fields.
measurement_identity_definitions = [
    (
        "measurement_id",
        "text",
        True,
        "Identifier formed as experiment_id::row_<source_excel_row>.",
    ),
    (
        "experiment_id",
        "text",
        True,
        "Link to the experiment containing this observation.",
    ),
    (
        "source_excel_row",
        "integer",
        True,
        "Original Excel row number, counted from 1.",
    ),
    (
        "measurement_point",
        "integer",
        True,
        "Source-reported Meas. Pts. value; not elapsed time.",
    ),
]

measurement_identity_dictionary = pd.DataFrame(
    measurement_identity_definitions,
    columns=[
        "field_name",
        "proposed_data_type",
        "required",
        "definition",
    ],
)

print("DRAFT MEASUREMENT IDENTITY DICTIONARY")
print(f"Fields documented: {len(measurement_identity_dictionary)}")

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
):
    display(measurement_identity_dictionary)

DRAFT MEASUREMENT IDENTITY DICTIONARY
Fields documented: 4


,field_name,proposed_data_type,required,definition
0,measurement_id,text,True,Identifier formed as experiment_id::row_<source_excel_row>.
1,experiment_id,text,True,Link to the experiment containing this observation.
2,source_excel_row,integer,True,"Original Excel row number, counted from 1."
3,measurement_point,integer,True,Source-reported Meas. Pts. value; not elapsed time.


## Rheological Variable Definitions

The proposed variable dictionary maps audited source headings to
consistent field names, standardised units and scientific definitions.
Together with the measurement-identity fields, it describes the
intended contents of the measurement table.

<details>
<summary><strong>Source coverage, units and interpretation</strong></summary>

### Variables available by test type

Both test types contain angular frequency, storage modulus, loss
modulus and temperature. The time-sweep workbooks additionally contain
shear stress, strain and complex viscosity.

When the two test types are combined, variables absent from a source
layout will remain missing for those records. Structural absence
will not be represented as zero.

### Unit conventions

Angular frequency will remain in rad/s. Strain will remain in percent:
a reported value of 1 represents 1%, not a strain fraction of 1.

The unit-label variants `[ｰC]` and `[Paｷs]` have documented mappings
to `[°C]` and `[Pa·s]` in the audit. These are label standardisations,
not numerical conversions. Original unit labels will remain traceable
through the audit metadata.

### Source-reported and calculated quantities

All seven variables are reported in the source workbooks. They are
not quantities calculated by this workflow. Source-reported values
may themselves have been derived by the instrument software; their
presence in Excel does not establish direct measurement.

Complex viscosity is retained as a source-reported rheological
quantity. No complex viscosity, loss tangent or other derived
property is calculated in this cell.

Temperature and strain entries describe the values recorded at
measurement level. They remain distinct from experimental settings
reported in the dataset description.

### Specification status and execution

The dictionary proposes numerical fields and records their source
coverage. It does not yet enforce value ranges, missing-value rules
or scientific consistency checks.

The cell requires pandas and IPython and can run independently of
earlier in-memory tables. It creates `rheology_variable_dictionary`
in memory without extracting measurements, modifying source files
or saving an output file.

</details>

In [15]:
import pandas as pd
from IPython.display import display

# Define proposed names, units and meanings from the audited source fields.
rheology_variable_definitions = [
    (
        "Angular Frequency",
        "angular_frequency_rad_s",
        "rad/s",
        "frequency_sweep | time_sweep",
        "Angular frequency of the applied oscillation.",
    ),
    (
        "Storage Modulus",
        "storage_modulus_pa",
        "Pa",
        "frequency_sweep | time_sweep",
        "G′: elastic component of the oscillatory shear response.",
    ),
    (
        "Loss Modulus",
        "loss_modulus_pa",
        "Pa",
        "frequency_sweep | time_sweep",
        "G″: dissipative component of the oscillatory shear response.",
    ),
    (
        "Temperature",
        "temperature_c",
        "°C",
        "frequency_sweep | time_sweep",
        "Temperature recorded for the measurement point.",
    ),
    (
        "Shear Stress",
        "shear_stress_pa",
        "Pa",
        "time_sweep",
        "Shear stress reported for the measurement point.",
    ),
    (
        "Strain",
        "strain_percent",
        "%",
        "time_sweep",
        "Reported strain expressed as a percentage.",
    ),
    (
        "Complex Viscosity",
        "complex_viscosity_pa_s",
        "Pa·s",
        "time_sweep",
        "Source-reported complex viscosity for the oscillatory test.",
    ),
]

rheology_variable_dictionary = pd.DataFrame(
    rheology_variable_definitions,
    columns=[
        "source_heading",
        "field_name",
        "standardised_unit",
        "source_test_types",
        "definition",
    ],
)

rheology_variable_dictionary.insert(
    2, "proposed_data_type", "number"
)
rheology_variable_dictionary["value_origin"] = "source_reported"

print("DRAFT RHEOLOGICAL VARIABLE DICTIONARY")
print(f"Variables documented: {len(rheology_variable_dictionary)}")
print("Measurement values transformed: 0")
print("Files written: 0")

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
):
    display(rheology_variable_dictionary)

DRAFT RHEOLOGICAL VARIABLE DICTIONARY
Variables documented: 7
Measurement values transformed: 0
Files written: 0


,source_heading,field_name,proposed_data_type,standardised_unit,source_test_types,definition,value_origin
0,Angular Frequency,angular_frequency_rad_s,number,rad/s,frequency_sweep | time_sweep,Angular frequency of the applied oscillation.,source_reported
1,Storage Modulus,storage_modulus_pa,number,Pa,frequency_sweep | time_sweep,G′: elastic component of the oscillatory shear response.,source_reported
2,Loss Modulus,loss_modulus_pa,number,Pa,frequency_sweep | time_sweep,G″: dissipative component of the oscillatory shear response.,source_reported
3,Temperature,temperature_c,number,°C,frequency_sweep | time_sweep,Temperature recorded for the measurement point.,source_reported
4,Shear Stress,shear_stress_pa,number,Pa,time_sweep,Shear stress reported for the measurement point.,source_reported
5,Strain,strain_percent,number,%,time_sweep,Reported strain expressed as a percentage.,source_reported
6,Complex Viscosity,complex_viscosity_pa_s,number,Pa·s,time_sweep,Source-reported complex viscosity for the oscillatory test.,source_reported


## Combined Measurement Data Dictionary

The identity and rheological field definitions are combined into one
draft measurement dictionary. Each field is assigned an expected
test-type scope and a missing-value policy.

<details>
<summary><strong>Field coverage and missing-value interpretation</strong></summary>

### Expected fields

The four identity fields, angular frequency, storage modulus, loss
modulus and temperature are expected for both test types.

Shear stress, strain and complex viscosity are expected only in
time-sweep source records. In a combined measurement table, these
three fields will remain missing for frequency-sweep records because
the source workbooks do not report them.

### Missing-value policy

Missing required identifiers or source-row locations must be resolved
before measurement records can be reliably linked and traced.

Missing source-reported values in otherwise expected fields will be
flagged for review. Observations will not be silently deleted or
filled with zero. Structural absence will be distinguished from
unexpected missing values using the experiment's test type.

### Execution and specification status

The cell uses `measurement_identity_dictionary` and
`rheology_variable_dictionary` from the preceding cells. It combines
their definitions and checks that the resulting field names are
unique and non-missing.

The result, `measurement_data_dictionary`, is an in-memory
specification. It does not extract measurements, enforce these
missing-value policies or save files. After a kernel restart, run
the two preceding dictionary-construction cells before this cell.

</details>

In [17]:
import pandas as pd
from IPython.display import display

both_test_types = "frequency_sweep | time_sweep"

# Prepare identity and provenance definitions without altering the originals.
identity_fields = measurement_identity_dictionary[
    ["field_name", "proposed_data_type", "definition"]
].copy()

identity_fields["standardised_unit"] = "Not applicable"
identity_fields["expected_in_test_types"] = both_test_types
identity_fields["missing_value_policy"] = (
    "Required for record identification or traceability; resolve before ingestion."
)

identity_fields.loc[
    identity_fields["field_name"].eq("measurement_point"),
    "missing_value_policy",
] = "Expected source value; flag missing values for review without filling."

# Prepare the rheological variable definitions.
rheology_fields = rheology_variable_dictionary[
    [
        "field_name",
        "proposed_data_type",
        "standardised_unit",
        "source_test_types",
        "definition",
    ]
].copy()

rheology_fields = rheology_fields.rename(
    columns={"source_test_types": "expected_in_test_types"}
)

rheology_fields["missing_value_policy"] = (
    "Expected source value; flag missing values for review without filling."
)

rheology_fields.loc[
    rheology_fields["expected_in_test_types"].eq("time_sweep"),
    "missing_value_policy",
] = (
    "Frequency sweep: structurally absent. "
    "Time sweep: flag missing values for review. Do not fill with zero."
)

# Combine the four identity fields and seven rheological fields.
dictionary_columns = [
    "field_name",
    "proposed_data_type",
    "standardised_unit",
    "expected_in_test_types",
    "missing_value_policy",
    "definition",
]

measurement_data_dictionary = pd.concat(
    [
        identity_fields[dictionary_columns],
        rheology_fields[dictionary_columns],
    ],
    ignore_index=True,
)

# Check the dictionary structure, not the experimental measurements.
field_names = measurement_data_dictionary["field_name"]

if field_names.isna().any() or not field_names.is_unique:
    raise ValueError("Dictionary field names must be non-missing and unique.")

print("COMBINED MEASUREMENT DATA DICTIONARY")
print(f"Fields documented: {len(measurement_data_dictionary)}")
print(f"Unique field names: {field_names.nunique()}")
print("Files written: 0")

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
):
    display(measurement_data_dictionary)

COMBINED MEASUREMENT DATA DICTIONARY
Fields documented: 11
Unique field names: 11
Files written: 0


,field_name,proposed_data_type,standardised_unit,expected_in_test_types,missing_value_policy,definition
0,measurement_id,text,Not applicable,frequency_sweep | time_sweep,Required for record identification or traceability; resolve before ingestion.,Identifier formed as experiment_id::row_<source_excel_row>.
1,experiment_id,text,Not applicable,frequency_sweep | time_sweep,Required for record identification or traceability; resolve before ingestion.,Link to the experiment containing this observation.
2,source_excel_row,integer,Not applicable,frequency_sweep | time_sweep,Required for record identification or traceability; resolve before ingestion.,"Original Excel row number, counted from 1."
3,measurement_point,integer,Not applicable,frequency_sweep | time_sweep,Expected source value; flag missing values for review without filling.,Source-reported Meas. Pts. value; not elapsed time.
4,angular_frequency_rad_s,number,rad/s,frequency_sweep | time_sweep,Expected source value; flag missing values for review without filling.,Angular frequency of the applied oscillation.
5,storage_modulus_pa,number,Pa,frequency_sweep | time_sweep,Expected source value; flag missing values for review without filling.,G′: elastic component of the oscillatory shear response.
6,loss_modulus_pa,number,Pa,frequency_sweep | time_sweep,Expected source value; flag missing values for review without filling.,G″: dissipative component of the oscillatory shear response.
7,temperature_c,number,°C,frequency_sweep | time_sweep,Expected source value; flag missing values for review without filling.,Temperature recorded for the measurement point.
8,shear_stress_pa,number,Pa,time_sweep,Frequency sweep: structurally absent. Time sweep: flag missing values for review. Do not fill with zero.,Shear stress reported for the measurement point.
9,strain_percent,number,%,time_sweep,Frequency sweep: structurally absent. Time sweep: flag missing values for review. Do not fill with zero.,Reported strain expressed as a percentage.


## Measurement Data Dictionary Export

The draft measurement data dictionary is saved to
`metadata/measurement_data_dictionary.csv`. It documents the 11 proposed
measurement fields, their data types, standardised units, expected test
types and missing-value policies.

<details>
<summary><strong>Execution dependencies and scope</strong></summary>

This export requires the preceding dictionary-construction cells to have
been executed in the current kernel. When reopening the notebook, run
the cells in order to recreate `measurement_data_dictionary` before
exporting it.

The CSV contains schema definitions, not experimental measurements.
The documented policies remain proposed processing rules; exporting
them does not apply them to the source data.

Re-running this cell replaces the exported dictionary with the current
in-memory version. Original workbooks and audit inventories remain
unchanged.

</details>

In [19]:
from pathlib import Path

# Recreate the project path independently of earlier path variables.
current_folder = Path.cwd().resolve()
project_root = (
    current_folder.parent
    if current_folder.name.lower() in {"notebook", "notebooks"}
    else current_folder
)

# Require the dictionary created by the preceding notebook cells.
if "measurement_data_dictionary" not in globals():
    raise RuntimeError(
        "Run the preceding dictionary-construction cells before this export."
    )

# Use the existing project metadata folder.
metadata_folder = project_root / "metadata"

if not metadata_folder.is_dir():
    raise FileNotFoundError(
        f"Metadata folder not found: {metadata_folder}\n"
        "Open this notebook from the project root or its notebooks folder."
    )

output_path = metadata_folder / "measurement_data_dictionary.csv"

# Save field definitions without adding the pandas row index.
measurement_data_dictionary.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig",
)

print("MEASUREMENT DATA DICTIONARY EXPORT")
print(f"Saved to: {output_path}")
print(f"Fields documented: {len(measurement_data_dictionary)}")
print("Files written: 1")

MEASUREMENT DATA DICTIONARY EXPORT
Saved to: C:\Users\sufer\Documents\collagen-hydrogel-rheology-data-curation\metadata\measurement_data_dictionary.csv
Fields documented: 11
Files written: 1


## Experiment Register and Data Dictionary Export

The draft experiment register and its field definitions are saved to
`metadata/experiment_register.csv` and
`metadata/experiment_data_dictionary.csv`, respectively.

The export uses `experiment_register_standardised`, in which replicate
labels are represented as text. Each experiment retains its dataset
label, source workbook filename and worksheet name.

<details>
<summary><strong>Export checks and reuse</strong></summary>

Before writing, the code checks that the register contains all documented
fields, that experiment identifiers are non-missing and unique, and that
replicate labels are non-empty text values. These structural checks do
not confirm shared physical hydrogel identity across test types.

Experiment identifiers remain stable while the dataset label, source
filename and worksheet name remain unchanged.

CSV files do not preserve pandas data types. When reloading the register,
`replicate_id` must be explicitly read as text, for example using
`dtype={"replicate_id": "string"}`.

This cell requires the preceding register-standardisation and experiment
data-dictionary cells to have been executed. Re-running it replaces the
two exports with their current in-memory versions.

</details>

In [21]:
from pathlib import Path

# Locate the project folder.
current_folder = Path.cwd().resolve()
project_root = (
    current_folder.parent
    if current_folder.name.lower() in {"notebook", "notebooks"}
    else current_folder
)

# Check that the preceding notebook cells have been run.
required_objects = [
    "experiment_register_standardised",
    "experiment_data_dictionary",
]

for object_name in required_objects:
    if object_name not in globals():
        raise RuntimeError(
            f"{object_name} is unavailable. "
            "Run the preceding notebook cells in order."
        )

metadata_folder = project_root / "metadata"

if not metadata_folder.is_dir():
    raise FileNotFoundError(
        f"Metadata folder not found: {metadata_folder}"
    )

# Export the documented fields in their dictionary order.
documented_fields = experiment_data_dictionary["field_name"].tolist()

missing_fields = [
    field
    for field in documented_fields
    if field not in experiment_register_standardised.columns
]

if missing_fields:
    raise ValueError(f"Register fields missing: {missing_fields}")

register_to_export = experiment_register_standardised[
    documented_fields
].copy()

# Check identifiers and the standardised replicate-label representation.
experiment_ids = register_to_export["experiment_id"]

if (
    experiment_ids.isna().any()
    or experiment_ids.astype("string").str.strip().eq("").any()
    or not experiment_ids.is_unique
):
    raise ValueError(
        "Experiment identifiers must be non-missing, non-blank and unique."
    )

replicate_labels = register_to_export["replicate_id"]

if not replicate_labels.map(
    lambda value: isinstance(value, str) and bool(value.strip())
).all():
    raise ValueError("Replicate labels must be non-empty text values.")

# Save the register and its field definitions.
register_path = metadata_folder / "experiment_register.csv"
dictionary_path = metadata_folder / "experiment_data_dictionary.csv"

register_to_export.to_csv(
    register_path,
    index=False,
    encoding="utf-8-sig",
)

experiment_data_dictionary.to_csv(
    dictionary_path,
    index=False,
    encoding="utf-8-sig",
)

print("EXPERIMENT REGISTER AND DATA DICTIONARY EXPORT")
print(f"Experiment records saved: {len(register_to_export)}")
print(f"Fields documented: {len(experiment_data_dictionary)}")
print(f"Register saved to: {register_path}")
print(f"Dictionary saved to: {dictionary_path}")
print("Files written: 2")

EXPERIMENT REGISTER AND DATA DICTIONARY EXPORT
Experiment records saved: 20
Fields documented: 7
Register saved to: C:\Users\sufer\Documents\collagen-hydrogel-rheology-data-curation\metadata\experiment_register.csv
Dictionary saved to: C:\Users\sufer\Documents\collagen-hydrogel-rheology-data-curation\metadata\experiment_data_dictionary.csv
Files written: 2


## Schema Relationships and Implementation Status

The schema separates experiment records from measurement observations
and quality-control findings. This supports source traceability while
preserving uncertainty about physical sample identity.

### Relationships Between Records

| Relationship | Linking information | Current status |
|---|---|---|
| Experiment to source worksheet | `dataset_id`, `original_filename` and `sheet_name` | Recorded in the exported experiment register. |
| Measurement to experiment | `experiment_id` | Defined in the measurement dictionary; measurement records have not yet been constructed. |
| Measurement to source row | `experiment_id` and `source_excel_row` | Defined for row-level provenance; measurement identifiers have not yet been generated. |
| Quality-control finding to source location | Source workbook, worksheet and affected cell or range | Retained in the audit issue log; links to processed measurements remain to be implemented. |
| Experiment to physical hydrogel | A sample identifier supported by evidence | Deferred because shared physical identity across test types remains unconfirmed. |

<details>
<summary><strong>Sample identity and quality-control linkage</strong></summary>

Matching concentration and replicate labels identify candidate
cross-test relationships, but do not establish that the tests used the
same physical hydrogel. No shared physical sample identifiers have been
assigned on this basis.

The physical hydrogel remains the experimental unit. Measurement points
within an experiment must not be counted as independent hydrogel samples.

Quality-control findings may concern an individual measurement, a range
of measurements or worksheet-level metadata. Their original source
locations will therefore remain available even where a direct link to
one processed measurement is inappropriate.

</details>

### Exported Outputs

| File | Contents |
|---|---|
| `metadata/experiment_register.csv` | 20 experiment records with source identifiers and standardised experiment metadata. |
| `metadata/experiment_data_dictionary.csv` | Definitions of the seven experiment-register fields. |
| `metadata/measurement_data_dictionary.csv` | Definitions of the 11 proposed measurement fields, including units, test-type applicability and missing-value policies. |

### Implementation Boundary

The experiment register and data dictionaries were constructed and
exported. The measurement table, measurement-level links to quality-control
findings and enforcement of the proposed measurement rules remain
unfinished.

Time-sweep measurement-point order will remain distinct from elapsed
time unless supporting timing information becomes available.

A final execution from a fresh kernel and review of the exported files
are required before this notebook is considered ready for release.

The notebook was rerun from a fresh kernel without reported execution
errors. The three exported tables were reviewed and matched the outputs
regenerated from the notebook's processing code and audit inventories.
These checks establish execution and export consistency for the current
inputs; they do not validate the underlying rheological measurements.